In [3]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import sys
sys.path.append('..')
from core.scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [5]:
links = fetch_website_links("https://savvyalgostudio.com")
links

['/',
 '/portfolio',
 '/about',
 '/blog',
 '/contact',
 'tel:+13073748824',
 '/contact',
 '/contact',
 '/portfolio',
 'tel:+13073748824',
 '#services',
 '#services',
 '/services/ai-call-center',
 '/portfolio/travel-24-7-revenue-pipeline',
 '/services/ai-call-center',
 '/services/agentic-ai',
 '/services/member-and-customer-retention',
 '/services/rag-chatbot-and-internal-ai',
 '/services/chargeback-and-compliance-ai',
 '/services/workflow-automation',
 '/services',
 '/portfolio',
 'tel:+13073748824',
 '/contact',
 'mailto:sales@savvyalgostudio.com',
 '/',
 '/services/ai-call-center',
 '/services/agentic-ai',
 '/services/member-and-customer-retention',
 '/services/rag-chatbot-and-internal-ai',
 '/services/chargeback-and-compliance-ai',
 '/services/workflow-automation',
 '/services/custom-crm-development',
 '/services/custom-app-development',
 '/services/cloud-infrastructure',
 '/services/system-integrations',
 '/industries/travel-timeshare',
 '/industries/retail',
 '/industries/medicare

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to inlcude in a big brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links":[
        {"type": "about page", "url":"https://savvyalgostudio.com/about"},
        {"type": "portfolio page", "url":"https://savvyalgostudio.com/portfolio"},
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt=f"""
    Here is the list of links on the website {url} - 
    Decide which of these are relavent web links for a brochure about the company,
    respond with the full https URL in JSON format.
    
    Links (some might be relative links):
    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt
    

In [8]:
print(get_links_user_prompt("https://savvyalgostudio.com"))


    Here is the list of links on the website https://savvyalgostudio.com - 
    Decide which of these are relavent web links for a brochure about the company,
    respond with the full https URL in JSON format.

    Links (some might be relative links):
    /
/portfolio
/about
/blog
/contact
tel:+13073748824
/contact
/contact
/portfolio
tel:+13073748824
#services
#services
/services/ai-call-center
/portfolio/travel-24-7-revenue-pipeline
/services/ai-call-center
/services/agentic-ai
/services/member-and-customer-retention
/services/rag-chatbot-and-internal-ai
/services/chargeback-and-compliance-ai
/services/workflow-automation
/services
/portfolio
tel:+13073748824
/contact
mailto:sales@savvyalgostudio.com
/
/services/ai-call-center
/services/agentic-ai
/services/member-and-customer-retention
/services/rag-chatbot-and-internal-ai
/services/chargeback-and-compliance-ai
/services/workflow-automation
/services/custom-crm-development
/services/custom-app-development
/services/cloud-infrastr

In [9]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system", "content": link_system_prompt},
            {"role":"user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )

    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [10]:
select_relevant_links("https://savvyalgostudio.com")

{'links': [{'type': 'about page', 'url': 'https://savvyalgostudio.com/about'},
  {'type': 'portfolio page', 'url': 'https://savvyalgostudio.com/portfolio'},
  {'type': 'contact page', 'url': 'https://savvyalgostudio.com/contact'},
  {'type': 'linkedin profile',
   'url': 'https://linkedin.com/company/savvyalgostudio'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://savvyalgostudio.com")

Selecting relevant links for https://savvyalgostudio.com by calling gpt-5-nano
Found 26 relevant links


{'links': [{'type': 'about page', 'url': 'https://savvyalgostudio.com/about'},
  {'type': 'portfolio page', 'url': 'https://savvyalgostudio.com/portfolio'},
  {'type': 'case study page',
   'url': 'https://savvyalgostudio.com/portfolio/travel-24-7-revenue-pipeline'},
  {'type': 'service page', 'url': 'https://savvyalgostudio.com/services'},
  {'type': 'service page',
   'url': 'https://savvyalgostudio.com/services/ai-call-center'},
  {'type': 'service page',
   'url': 'https://savvyalgostudio.com/services/agentic-ai'},
  {'type': 'service page',
   'url': 'https://savvyalgostudio.com/services/member-and-customer-retention'},
  {'type': 'service page',
   'url': 'https://savvyalgostudio.com/services/rag-chatbot-and-internal-ai'},
  {'type': 'service page',
   'url': 'https://savvyalgostudio.com/services/chargeback-and-compliance-ai'},
  {'type': 'service page',
   'url': 'https://savvyalgostudio.com/services/workflow-automation'},
  {'type': 'service page',
   'url': 'https://savvyalgos

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://savvyalgostudio.com"))

Selecting relevant links for https://savvyalgostudio.com by calling gpt-5-nano
Found 27 relevant links
## Landing Page:

Savvy Algo Studio — Growth-Focused AI Agency

AI Systems & Digitalization for Operations-Driven Businesses
We Digitalize Your Operations.
Then We Make Them 
Intelligent.
We build the systems your business runs on — your CRM, your call center, your follow-up — then add the AI that makes them perform. You own all of it.
Book a Free 30-Min Strategy Call 
See What We've Built
Or call us now — 
+1 (307) 374-8824
 · We answer around the clock
7+ years in production
·
Live across the UK, US, GCC, EU, Canada & Australia
·
Built for you, owned by you
AI Call Center · Live
Inbound desk
ACTIVE
22
live calls
71%
containment
+0.7
CSAT
CRM activity
just now
 
Lead qualified
+$1,840 pipeline
 
Member reactivated
tier: gold
 
Chargeback rebuttal sent
auto-filed
Retention rate
+18.4% MoM
▲ rising
Sound Familiar?
Where Your Business Is Losing Money Right Now
You're not short on ambiti

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("Savvy Algo Studio", "https://savvyalgostudio.com")

Selecting relevant links for https://savvyalgostudio.com by calling gpt-5-nano
Found 27 relevant links


"\nYou are looking at a company called: Savvy Algo Studio\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nSavvy Algo Studio — Growth-Focused AI Agency\n\nAI Systems & Digitalization for Operations-Driven Businesses\nWe Digitalize Your Operations.\nThen We Make Them \nIntelligent.\nWe build the systems your business runs on — your CRM, your call center, your follow-up — then add the AI that makes them perform. You own all of it.\nBook a Free 30-Min Strategy Call \nSee What We've Built\nOr call us now — \n+1 (307) 374-8824\n · We answer around the clock\n7+ years in production\n·\nLive across the UK, US, GCC, EU, Canada & Australia\n·\nBuilt for you, owned by you\nAI Call Center · Live\nInbound desk\nACTIVE\n22\nlive calls\n71%\ncontainment\n+0.7\nCSAT\nCRM activity\njust now\n \nLead qualified\n+$1,840 pipeline\n \nMember reactivated\ntier: gold\n

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [30]:
create_brochure("Savvy Algo Studio", "https://savvyalgostudio.com")

Selecting relevant links for https://savvyalgostudio.com by calling gpt-5-nano
Found 25 relevant links


# Savvy Algo Studio  
**Growth-Focused AI Agency**  

---

## Who We Are  
Savvy Algo Studio specializes in building AI-driven operational systems tailored for operations-driven businesses. With over 7 years of experience, we digitalize your core processes — from CRM and call centers to follow-up mechanisms — and supercharge them with intelligent AI overlays. What's unique? You own the systems outright.  

Our origins lie in building the backbone of a multi-brand travel business, delivering production-ready AI systems that solve real business problems — not mere prototypes or pilots. Today, we serve clients worldwide across the UK, US, GCC, EU, Canada, and Australia with a full suite of AI-enhanced digital solutions.  

---

## What We Offer  
- **AI-Powered CRM Systems:** Custom-built CRMs designed to suit how you actually sell, integrating data that previously lived in disconnected tools.  
- **AI Call Centers:** Automated, intelligent inbound call centers operating 24/7 to improve containment rates and customer satisfaction.  
- **Automated Retention & Chargeback Defense:** Systems to proactively retain members and handle disputes with minimal human intervention.  
- **Full Operational Digitalization:** We build and connect your operational tools end-to-end, preventing revenue leakage through disconnected systems or manual processes.  

---

## Why Choose Savvy Algo Studio  
- **Proven Production Systems:** Over 7 years of running live AI agents and more than 100 custom apps delivered.  
- **Global Reach:** Active projects and live deployments on multiple continents including North America, Europe, and the Middle East.  
- **Built for You, Owned by You:** Complete ownership of the systems we build to empower your long-term growth.  
- **Outcome-Driven:** Founded by operators who understand real business outcomes and operational efficiency.  

---

## The Challenges We Solve  
- Manual, repetitive tasks consuming your team’s time that systems should handle automatically.  
- Fragmented customer data scattered across multiple tools that don’t communicate.  
- Tools and processes that leak revenue due to lack of integration or automation.  
- Uncertainty about how or where to begin using AI effectively and cost-efficiently.  
- Inconsistent customer experience based on which agent answers the call, lacking a unified knowledge base or AI support.  

---

## Who We Serve  
Our AI systems are built for operations-driven companies looking to:  
- Streamline and digitize their workflows.  
- Harness AI to improve efficiency and customer experience.  
- Gain ownership and control over their digital operational infrastructure.  
Industries include travel, customer service, sales, compliance, and other sectors where operational excellence is key.  

---

## Company Culture & Careers  
Savvy Algo Studio is a team of operators, technologists, and problem-solvers passionate about building practical AI systems that deliver measurable business impact. We value:  
- **Collaboration:** Working closely with clients to truly understand their operational needs.  
- **Innovation:** Continuously developing new AI agents and custom apps.  
- **Ownership:** Empowering employees and clients alike with ownership mentality.  
- **Global Perspective:** Serving a diverse international clientele.  

Interested in joining a growth-minded AI agency with a proven track record? Savvy Algo Studio seeks talented professionals in AI development, system architecture, client strategy, and operations to help build the future of business automation.  

---

## Connect With Us  
Ready to digitalize and intelligently transform your operations?  
**Book a free 30-minute strategy call** or call us anytime at +1 (307) 374-8824 — we answer 24/7.  

See firsthand what we’ve built and how we can deliver transformational AI systems for your business.  

---

Savvy Algo Studio — *Where your business runs smarter, not harder.*

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [33]:
stream_brochure("Savvy Algo Studio", "https://savvyalgostudio.com")

Selecting relevant links for https://savvyalgostudio.com by calling gpt-5-nano
Found 23 relevant links


# Savvy Algo Studio Brochure

---

## Who We Are  
Savvy Algo Studio is a **growth-focused AI agency** specializing in building intelligent digital systems that run your business operations. With over **7 years of live production experience**, we deliver end-to-end AI-enhanced platforms such as CRMs, call centers, and follow-up systems—custom-built for your unique needs and fully owned by you.

We operate globally across six key markets including the UK, US, GCC, EU, Canada, and Australia, serving industries from travel and insurance to logistics, legal, retail, and education.

---

## What We Do  
- **Digitalize Your Operations:** Replace manual tasks and disconnected tools with seamless operational systems.  
- **Make Them Intelligent:** Add AI layers that act intelligently — from 24/7 AI call centers to automated lead qualification and retention workflows.  
- **Build for Outcomes:** Every system aims to increase sales, improve retention, and grow your revenue. If an AI system does not justify itself with a business outcome, we do not recommend it.

---

## Why Choose Savvy Algo Studio  
- **Proven Production Systems:** Over 100 custom apps and CRMs delivered with 20+ AI agents currently in development.  
- **Outcome-Focused:** Every project starts with measurable goals — no fluff or unnecessary features.  
- **Transparent Partnership:** Clear scopes, honest timelines, and open communication so you always know what we’re building and why.  
- **Enduring Support:** We don’t just build your system and walk away — if performance dips, we fix it.  
- **Deep Understanding:** We immerse ourselves in your industries, compliance requirements, and customer journeys—not just the code.

---

## Our Customers  
We serve operations-driven businesses worldwide struggling with:  
- Disjointed data across multiple tools  
- Manual lead chasing and follow-up  
- Inconsistent customer experiences dependent on individual agents  
- Lost revenue through operational gaps

Our AI systems ensure:  
- Automation of repetitive tasks that free your team’s time  
- Unified data architecture aligned with how you sell and operate  
- AI call centers that can handle up to 90% of support calls without humans  
- Consistent, high-quality customer engagement 24/7

---

## Company Culture  
At Savvy Algo Studio, our core values guide everything we do:  
- **Outcomes Over Everything:** Success is measured by business impact, never by tech for tech’s sake.  
- **The Right Tool, Not Every Tool:** We build only what you need, nothing unnecessary.  
- **Open and Honest Partnership:** Clear, straightforward communication with no surprises.  
- **We Stand Behind Our Work:** Ongoing commitment beyond delivery.  
- **Depth Over Width:** Deep operational understanding fuels our technical excellence.

Our team comprises experienced operators and AI specialists who know what it takes to build systems that actually run your business effectively.

---

## Careers at Savvy Algo Studio  
If you're passionate about building real-world AI systems that drive measurable business growth and are excited to work as part of a collaborative, transparent, and outcome-driven team, Savvy Algo Studio could be the perfect fit.

We value people who:  
- Thrive on solving complex operational problems  
- Prioritize clear communication and partnership  
- Want to create AI that delivers practical, lasting results

---

## Contact Us  
Book a **free 30-minute strategy call** to discuss how Savvy Algo Studio can transform your operations with AI.  
Phone: +1 (307) 374-8824 (available around the clock)  
Website: [savvyalgostudio.com](https://savvyalgostudio.com)

---

Savvy Algo Studio — **Your Partner in AI-Driven Revenue Growth and Operational Excellence**

In [1]:
import gradio as gr

In [ ]:
name_input = gr.Textbox(label="Company Name")
url_input = gr.Textbox(label="Website URL")
message_output = gr.Markdown(label="Respoonse")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator",
    inputs=[name_input, url_input],
    outputs=[message_output],
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Savvy Algo Studio", "https://savvyalgostudio.com", "GPT"]
        ], 
    flagging_mode ="never"
)
view.launch()


* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


Selecting relevant links for https://savvyalgostudio.com by calling gpt-5-nano
Found 23 relevant links


# Savvy Algo Studio Brochure

---

## About Savvy Algo Studio

**Savvy Algo Studio** is a growth-focused AI agency specializing in building intelligent AI systems and digitalization solutions for operations-driven businesses. With over **7 years of experience**, we transform your business operations by designing and implementing custom CRMs, AI-powered call centers, and automated follow-up systems to ensure seamless customer journeys and operational efficiency.

Our roots lie in real-world operations—we started by building comprehensive operational backbones like CRMs, dialers, payment gateways, and compliance systems for a multi-brand travel business. From there, we integrated AI to create live inbound call centers, automated retention programs, and chargeback defenses—all running 24/7 to serve businesses across the UK, US, GCC, EU, Canada, and Australia.

---

## What We Do

- **Digitalize Your Operations:** We build the robust systems your business runs on, letting your teams focus on growth instead of administrative overload.
- **Make Operations Intelligent:** We layer AI on top of your existing systems to automate lead qualification, customer follow-ups, retention, and disputed chargeback rebuttals.
- **Ownership:** You own all the systems we build—there are no rented platforms, no third-party lock-ins.
- **Live AI Call Centers:** Our AI-powered call centers are live and actively managing customer interactions, delivering real-time containment rates of 71% and driving measurable pipeline growth.

---

## Why Savvy Algo Studio?

Many businesses lose revenue due to:

- Teams manually chasing leads and logging calls instead of focusing on customer engagement.
- Customer data trapped in multiple disconnected tools.
- Tools that don’t communicate, causing leads, payments, and follow-ups to slip through the cracks.
- Inconsistent customer experiences depending on which agent answers calls.
- Uncertainty about where to begin with AI implementation or whom to trust to build effective AI solutions.

At Savvy Algo Studio, we solve these by creating connected, intelligent systems tailored exactly to your business processes—eliminating gaps and automating repetitive work.

---

## Our Impact & Reach

- 20+ AI agents currently in development.
- 100+ custom apps and CRM systems delivered.
- Operating live AI systems with consistent month-over-month retention rate improvements (+18.4% MoM).
- Serving multiple geographies including the UK, US, GCC, EU, Canada, and Australia.
- Available 24/7 with around-the-clock client support.

---

## Our Culture

We are **operators first**, driven by outcomes—not prototypes or pilots. Our team thrives on solving real business challenges with AI systems that run reliably in production every day. We value operational excellence, deep client collaboration, transparency, and ownership.

---

## Careers at Savvy Algo Studio

Join a team pioneering AI solutions with a strong operational foundation! We seek talented developers, AI engineers, product managers, and customer success specialists passionate about building live AI-driven business systems that deliver measurable growth.

- Work remotely or from hubs in multiple global regions.
- Be part of an outcome-focused culture.
- Engage with breakthrough AI products impacting diverse industries.

---

## Get In Touch

Ready to transform your operations and unlock growth with AI?

**Book a Free 30-Min Strategy Call**  
Or call us anytime at: +1 (307) 374-8824  
We answer around the clock.

Discover how Savvy Algo Studio can build and own your intelligent AI systems today.

---

Savvy Algo Studio — Making Your Business Operations Smarter and Scalable.